# CDR vs adversarial VLM — full run on Google Colab, resumable across accounts

Runs the MVP (`docs/proposal/proposal.tex`) on a Colab GPU, keeping **LLaVA-1.5-7B**.
The full 200-image x 3-epsilon attack exceeds one account's GPU quota, so the
notebook is built to **hit the limit, switch account, and continue** — not restart.

**State travels two ways, merged never overwritten (see `docs/implementation/colab-resume.md`):**
- **Path A (recommended):** share the `DRIVE_DIR` folder from the main account to the
  others with Editor rights; on each other account *Add shortcut to Drive* into My
  Drive, keeping the name. All accounts read/write one store.
- **Path B (fallback):** after each sync the notebook writes `cdr_state.zip` to
  `DRIVE_DIR`. Copy it into the new account's `DRIVE_DIR` and Run all.

**Resume layers:** (1) any finished job (`jobs/**.json`) is skipped; (2) start-of-session
pull from Drive + zip is merge-only; (3) only the small results/state live on Drive (the ~14 GB model and the dataset
re-download to ephemeral disk each session, which keeps the shared Drive folder under
its 15 GB free quota); (4) `--max-hours` stops cleanly before the runtime dies and
a background thread autosyncs every few minutes.

Set `Runtime > Change runtime type > GPU` (L4/A100 preferred), then `Runtime > Run all`.
The only cell you edit is CELL 1.

## CELL 1 — parameters + GPU check (the only cell you edit)

In [ ]:
import shutil, subprocess
if shutil.which("nvidia-smi") is None:
    print("[warn] No GPU. Runtime > Change runtime type > GPU, then Run all.")
else:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

# --- source ---
REPO_URL = "https://github.com/nongvantinh/computer-vision.git"   # <-- your repo
BRANCH   = "master"
RUN_ID   = "pilot_12"   # run dir name; "full_200" for the whole matrix

# --- folders ---
PROJECT_DIR = "/content/cdr"                                         # code + fast disk
DRIVE_DIR   = "/content/drive/MyDrive/computer-vision"              # durable store

# --- multi-account continuation ---
ACCOUNT_LABEL         = "main"        # change per account; only labels the session log
SESSION_HOURS         = 3.0           # stop cleanly before Colab reclaims the runtime
AUTOSYNC_MINUTES      = 5             # background push cadence
EXPECT_EXISTING_STATE = False         # True on 2nd+ account so an empty store is an ERROR
FORCE_UNLOCK          = False         # True only if a previous session is truly dead

# --- Dataset: the free 20-class imagenette+imagewoof subset is fetched automatically
#     to ephemeral disk each session (deterministic; identical 200 images every time).
#     Nothing to upload.

## CELL 2 — mount Drive, get the code, import the sync module

In [ ]:
import os, sys, time, json, threading
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
if shutil.which("rsync") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "rsync"], check=True)

DRIVE, PROJECT = Path(DRIVE_DIR), Path(PROJECT_DIR)

def sh(cmd, cwd=None, env=None, check=False, quiet=False):
    if isinstance(cmd, str): cmd = ["bash", "-lc", cmd]
    m = os.environ.copy(); m.update(env or {})
    p = subprocess.Popen([str(c) for c in cmd], cwd=str(cwd) if cwd else None, env=m,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet: print(line, end="")
    rc = p.wait()
    if check and rc: raise RuntimeError(f"failed ({rc}): {cmd}")
    return rc, "".join(out)

if (PROJECT / ".git").exists():
    sh(["git", "-C", str(PROJECT), "pull", "--ff-only"])
else:
    rc, _ = sh(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT)])
    if rc:
        arch = DRIVE / "source.zip"
        assert arch.exists(), f"clone failed; for a private repo put source.zip in {DRIVE}"
        PROJECT.mkdir(parents=True, exist_ok=True)
        sh(["unzip", "-q", "-o", str(arch), "-d", str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT))
from src.colab.sync import StateStore, merge_missing, merge_jsonl, LIGHT_FILTERS, count_done_jobs
print("[ok] code ready at", PROJECT)

## CELL 3 — Drive tree, state store, session lock, empty-state guard

In [ ]:
for sub in ("results", "state", "hf_cache", "imagenet", "reports"):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

# fail fast on Viewer-only shared folders
try:
    probe = DRIVE / "state" / ".write_probe"; probe.write_text(str(time.time())); probe.unlink()
except OSError as e:
    raise RuntimeError(f"cannot write to {DRIVE/'state'}: {e}\n"
                       "Shared folder must be Editor + 'Add shortcut to Drive'.")

store = StateStore(DRIVE / "state")
SESSION_ID = f"{ACCOUNT_LABEL}-{time.strftime('%Y%m%d_%H%M%S')}"
store.acquire_lock(SESSION_ID, ACCOUNT_LABEL, force=FORCE_UNLOCK)
store.append("sessions.jsonl", {"session": SESSION_ID, "account": ACCOUNT_LABEL,
                                "started": time.strftime("%Y-%m-%d %H:%M:%S")})
hist = store.history()
print(f"[state] {len(hist)} prior session(s); this session {SESSION_ID}")

RUNS_BASE_DRIVE = DRIVE / "results" / "runs"
RUNS_BASE_LOCAL = PROJECT / "results" / "runs"
DRIVE_RESULTS = RUNS_BASE_DRIVE / RUN_ID
LOCAL_RESULTS = RUNS_BASE_LOCAL / RUN_ID
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

## CELL 4 — restore state (Drive + zip, merge-only) and start autosync

In [ ]:
def rsync(src, dst, filters=(), quiet=True):
    Path(dst).mkdir(parents=True, exist_ok=True)
    return sh(["rsync", "-a", *filters, f"{src}/", f"{dst}/"], quiet=quiet)

# 1) pull light state Drive -> local (per-job JSON, tables, plots; not example PNGs)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
rsync(DRIVE_RESULTS, LOCAL_RESULTS, filters=LIGHT_FILTERS, quiet=False)

# 2) merge the fallback zip if present (Path B)
bundle = DRIVE / "cdr_state.zip"
if bundle.exists():
    stage = PROJECT.parent / "cdr_restore"; shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True)
    if sh(["unzip", "-q", str(bundle), "-d", str(stage)], quiet=True)[0] == 0:
        added = merge_missing(stage / "results" / "mvp", LOCAL_RESULTS)
        if (stage / "state").exists():
            merge_missing(stage / "state", DRIVE / "state", skip=("session.lock", "sessions.jsonl", ".write_probe"))
            merge_jsonl(stage / "state" / "sessions.jsonl", DRIVE / "state" / "sessions.jsonl")
        print(f"[zip] merged {added} missing job/result files")
    shutil.rmtree(stage, ignore_errors=True)

done = count_done_jobs(LOCAL_RESULTS)
print(f"[state] restored jobs: {done}")
if sum(done.values()) == 0 and EXPECT_EXISTING_STATE:
    raise RuntimeError("empty store but EXPECT_EXISTING_STATE=True: state did not follow you.\n"
                       "Use Path A (share folder + shortcut) or copy cdr_state.zip (Path B), then Run all.")

def write_bundle():
    stage = PROJECT.parent / "cdr_bundle"; shutil.rmtree(stage, ignore_errors=True)
    (stage / "results").mkdir(parents=True)
    # light only: the jobs/ tree + derived tables, no example PNGs / no model
    sh(["rsync", "-a", *LIGHT_FILTERS, f"{LOCAL_RESULTS}/", f"{stage/'results'/'mvp'}/"], quiet=True)
    shutil.copytree(DRIVE / "state", stage / "state",
                    ignore=shutil.ignore_patterns("session.lock", ".write_probe"))
    z = PROJECT.parent / "cdr_state.zip"; z.unlink(missing_ok=True)
    sh(["bash", "-lc", f"cd '{stage}' && zip -qr '{z}' ."], quiet=True)
    shutil.move(str(z), str(bundle)); shutil.rmtree(stage, ignore_errors=True)

def sync_to_drive(quiet=True):
    rsync(LOCAL_RESULTS, DRIVE_RESULTS, quiet=quiet)   # full push incl. example PNGs

_stop = threading.Event()
def _loop():
    while not _stop.wait(AUTOSYNC_MINUTES * 60):
        try:
            store.heartbeat(SESSION_ID, ACCOUNT_LABEL, jobs=count_done_jobs(LOCAL_RESULTS))
            sync_to_drive()
        except Exception as e:
            print("[autosync]", e)
if not globals().get("_autosync"):
    threading.Thread(target=_loop, daemon=True).start(); _autosync = True
print(f"[ok] autosync every {AUTOSYNC_MINUTES} min")

## CELL 5 — environment (uv), model + ImageNet on Drive (downloaded once)

In [ ]:
os.environ["UV_CACHE_DIR"] = "/content/.uv-cache"
os.environ["PATH"] = f"{Path.home()}/.local/bin:{os.environ['PATH']}"
if shutil.which("uv") is None:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh", check=True)
sh(["uv", "python", "install", "3.12"], cwd=PROJECT, check=True)
# CUDA torch to match Colab drivers, then the rest from pyproject
sh(["uv", "pip", "install", "--python", ".venv/bin/python", "torch",
    "--index-url", "https://download.pytorch.org/whl/cu124"], cwd=PROJECT, check=True)     if not (PROJECT/".venv").exists() else None
sh(["uv", "sync"], cwd=PROJECT, check=True)
PY = str(PROJECT / ".venv" / "bin" / "python")

# Free Drive is 15 GB and the model is ~14 GB, so the HF cache lives on ephemeral
# /content and the model re-downloads each session (~10 min). Only small results
# and state go to Drive, which keeps the shared folder well under quota.
HF_CACHE = "/content/hf_cache"
os.environ["HF_HOME"] = HF_CACHE
KT_ENV = {"HF_HOME": HF_CACHE, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg",
          "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

print("[ok] env ready:", PY)

## CELL 6 — pin the run profile (identical settings across accounts)

In [ ]:
# Lock the attack/eval config once so every account/GPU produces comparable numbers,
# exactly like the KT profile.json batch-size lock.
profile = store.read("profile.json")
if profile is None:
    import yaml
    exp = yaml.safe_load((PROJECT / "configs" / "experiment.yaml").read_text())
    atk = yaml.safe_load((PROJECT / "configs" / "attack.yaml").read_text())
    profile = {"model": exp["model"]["model_id"], "dtype": exp["model"]["dtype"],
               "epsilon": atk["epsilon"], "steps": atk["steps"],
               "step_size": atk["step_size"], "seed": exp["seed"],
               "locked_by": SESSION_ID, "locked_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    store.write("profile.json", profile)
    print("[profile] locked:", profile)
else:
    print("[profile] reusing locked profile from", profile.get("locked_by"), ":", profile)

## CELL 7 — build dataset manifest, then checkpoint 1 (LLaVA gradient)

In [ ]:
# Fetch the free 20-class subset (imagenette+imagewoof) to ephemeral disk. The
# selection is deterministic (fixed wnids + seed), so every account/session builds
# the identical 200-image manifest. Only results/state persist to Drive.
sh([PY, "scripts/fetch_imagenette.py", "--out", "data/imagenet/val"],
   cwd=PROJECT, env=KT_ENV, check=True)

sh([PY, "scripts/build_dataset.py", "--config", "configs/experiment.yaml"], cwd=PROJECT, env=KT_ENV, check=True)
sh([PY, "scripts/test_llava.py"], cwd=PROJECT, env=KT_ENV)   # CHECKPOINT 1

## CELL 8 — run the experiment (resumable, time-budgeted)

In [ ]:
# The stable dir results/mvp is what autosync mirrors to Drive; a killed session
# resumes by skipping finished jobs. --max-hours stops cleanly within the budget.
sh([PY, "scripts/run_experiment.py",
    "--run-id", RUN_ID, "--runs-base", str(RUNS_BASE_LOCAL),
    "--max-hours", str(SESSION_HOURS)],
   cwd=PROJECT, env=KT_ENV)
sync_to_drive(quiet=False); write_bundle()
print("[state] jobs done:", count_done_jobs(LOCAL_RESULTS))

## CELL 9 — analysis, plots, demo sheets, final handover

In [ ]:
sh([PY, "scripts/run_analysis.py", "--run", str(LOCAL_RESULTS)], cwd=PROJECT, env=KT_ENV)
sh([PY, "-m", "src.visualization.demo", "--run", str(LOCAL_RESULTS), "--static", "--limit", "8"],
   cwd=PROJECT, env=KT_ENV)

# final push + release lock so another account can take over cleanly
sync_to_drive(quiet=False); write_bundle()
_stop.set(); store.release_lock()
done = count_done_jobs(LOCAL_RESULTS)
print("=" * 60)
print(f"session {SESSION_ID} handed over")
print(f"  jobs done (cumulative, all accounts): {done}")
print(f"  durable run dir: {DRIVE_RESULTS}")
print(f"  fallback zip   : {bundle}")
summ = DRIVE_RESULTS / "summary.json"
if summ.exists():
    cov = json.loads(summ.read_text())
    print(f"  coverage: {cov.get('n_attack_jobs')} attack, "
          f"{cov.get('n_defense_ok')} defense ok, {cov.get('n_defense_failed')} failed")
print("\nNot complete? Continue on another account: open this notebook there and Run all")
print("(Path A: shared folder, nothing to copy. Path B: copy cdr_state.zip into its DRIVE_DIR).")
print("Complete? Pull summary.json + statistics.json + plots/ from the run dir for the report.")